# Wallerina — API Validation Notebook

Corrected replacement for `reference.ipynb`.

`reference.ipynb` proved the endpoints *respond*. This notebook proves they
return **usable** data, by asserting on the results instead of eyeballing prints.

Fixes applied:

| # | Problem in reference.ipynb | Fix here |
|---|---|---|
| 1 | Native ETH showed `Unknown` (decimals is `None`) | 18-decimal fallback for native tokens |
| 2 | `withPrices: True` requested but never read | Price parser + USD valuation + assertions |
| 3 | Spam token with `uint256` max balance | Spam filter before any ratio math |
| 4 | Stable/volatile split never exercised | Explicit classifier + registry self-check |
| 5 | `change_5m` always `+0.00%` | 24h window, `fidelity=60` |
| 6 | `wallet_assets` referenced but never defined | Built from the wallet step |
| 7 | Per-asset `if/elif` relevance chain | Config-driven `ASSET_CONFIG` |
| 8 | `/events` keyword scan (500 -> 7 hits) | Dropped; `/public-search` only |
| 9 | Probabilities used as loose "signals" | Parsed into a market-implied CDF per expiry |

Run top to bottom. Every section ends in a `CHECK` block that raises on failure.


In [ ]:
import os, sys, subprocess

try:
    import requests
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "requests"], check=True)
    import requests

import re, json, time
from decimal import Decimal
from collections import defaultdict, Counter

# --------------------------------------------------------------
# CONFIG
# --------------------------------------------------------------
# The key is NOT hardcoded. reference.ipynb committed a live Alchemy key
# to git — rotate that one and load from the environment instead.
#
#   Windows PowerShell:  $env:ALCHEMY_API_KEY = "alch_..."
#   bash:                export ALCHEMY_API_KEY="alch_..."

ALCHEMY_API_KEY = os.environ.get("ALCHEMY_API_KEY")

if not ALCHEMY_API_KEY:
    import getpass
    ALCHEMY_API_KEY = getpass.getpass("ALCHEMY_API_KEY: ").strip()

# Swap this for a wallet that actually holds USDC/USDT, otherwise the
# stablecoin branch of the classifier stays untested (that was the case
# in reference.ipynb).
WALLET_ADDRESS = os.environ.get(
    "WALLET_ADDRESS",
    "0xde0B295669a9FD93d5F28D9Ec85E40f4cb697BAe"
)

NETWORKS = ["eth-mainnet", "base-mainnet", "arb-mainnet", "polygon-mainnet"]

NATIVE_DECIMALS = 18
NATIVE_SYMBOL = {
    "eth-mainnet": "ETH",
    "base-mainnet": "ETH",
    "arb-mainnet": "ETH",
    "polygon-mainnet": "POL",
    "matic-mainnet": "POL",
}

print("key loaded:", bool(ALCHEMY_API_KEY))
print("wallet:", WALLET_ADDRESS)

## 1. Alchemy — raw fetch

Same call as `reference.ipynb` cell 2, but we keep the full objects instead of
printing three fields, and we dump one raw token so the actual price field
shape is visible rather than assumed.

In [ ]:
ALCHEMY_URL = (
    f"https://api.g.alchemy.com/data/v1/{ALCHEMY_API_KEY}"
    "/assets/tokens/by-address"
)

payload = {
    "addresses": [{"address": WALLET_ADDRESS, "networks": NETWORKS}],
    "withMetadata": True,
    "withPrices": True,
    "includeNativeTokens": True,
    "includeErc20Tokens": True,
}

response = requests.post(ALCHEMY_URL, json=payload, timeout=60)
response.raise_for_status()

body = response.json()
tokens = body["data"]["tokens"]

print("HTTP", response.status_code)
print("tokens returned:", len(tokens))

# ---- inspect the real shape of a priced token (do not assume) ----
sample = next(
    (t for t in tokens if t.get("tokenPrices")),
    tokens[0] if tokens else None,
)
print("\n--- RAW SAMPLE ---")
print(json.dumps(sample, indent=2)[:1200])

## 2. Decode balances

Two bugs from `reference.ipynb` cell 3 live here.

**Native tokens.** Alchemy returns `symbol: None` and `decimals: None` for the
chain's native asset. The old code caught the missing symbol but then hit
`decimals is None` and printed `Unknown` — silently dropping every native ETH
balance, the single most important holding in the wallet.

**Prices.** `withPrices: True` was requested and never read. Without USD values
there is no `portfolio_ratio`, and without that there is no allocation engine.

In [ ]:
def parse_price(token):
    """Return USD price as float, or None.

    Alchemy has shipped this field in more than one shape, so try the known
    ones rather than trusting a single path.
    """
    prices = token.get("tokenPrices") or token.get("prices") or []

    if isinstance(prices, dict):
        prices = [prices]

    for entry in prices:
        if not isinstance(entry, dict):
            continue
        currency = str(entry.get("currency", "usd")).lower()
        if currency not in ("usd", "usd_price", ""):
            continue
        try:
            return float(entry.get("value"))
        except (TypeError, ValueError):
            continue
    return None


def is_native(token):
    meta = token.get("tokenMetadata") or {}
    return (
        token.get("tokenAddress") in (None, "", "0x")
        or meta.get("symbol") is None
    )


def decode_balance(token):
    """Return (Decimal balance, decimals_used, used_fallback)."""
    meta = token.get("tokenMetadata") or {}
    raw_hex = token.get("tokenBalance") or "0x0"
    raw = int(raw_hex, 16)

    decimals = meta.get("decimals")
    used_fallback = False

    if decimals is None:
        # THE FIX: native assets are always 18 decimals.
        if is_native(token):
            decimals = NATIVE_DECIMALS
            used_fallback = True
        else:
            return None, None, False

    return Decimal(raw) / (Decimal(10) ** int(decimals)), int(decimals), used_fallback


rows = []
for token in tokens:
    meta = token.get("tokenMetadata") or {}
    network = token.get("network")
    raw = int((token.get("tokenBalance") or "0x0"), 16)

    if raw == 0:
        continue

    balance, decimals, fallback = decode_balance(token)
    if balance is None:
        rows.append({
            "network": network,
            "symbol": meta.get("symbol"),
            "name": meta.get("name"),
            "address": token.get("tokenAddress"),
            "raw": raw,
            "balance": None,
            "decimals": None,
            "native_fallback": False,
            "price_usd": parse_price(token),
            "value_usd": None,
            "undecodable": True,
        })
        continue

    symbol = meta.get("symbol") or NATIVE_SYMBOL.get(network, "NATIVE")
    price = parse_price(token)

    rows.append({
        "network": network,
        "symbol": symbol,
        "name": meta.get("name"),
        "address": token.get("tokenAddress"),
        "raw": raw,
        "balance": balance,
        "decimals": decimals,
        "native_fallback": fallback,
        "price_usd": price,
        "value_usd": float(balance) * price if price is not None else None,
        "undecodable": False,
    })

natives = [r for r in rows if r["native_fallback"]]
print(f"non-zero positions: {len(rows)}")
print(f"native positions decoded via fallback: {len(natives)}")

for r in natives:
    value = f"${r['value_usd']:,.2f}" if r["value_usd"] is not None else "(no price)"
    print(f"  {r['network']:<16} {r['symbol']:<6} "
          f"{r['balance']:>18,.6f}  {value}")

### CHECK — native decode

This is the assertion `reference.ipynb` would have failed.

In [ ]:
unknowns = [r for r in rows if r["undecodable"]]

assert natives, (
    "No native balance decoded. reference.ipynb printed 'Unknown' for all four "
    "chains here — if this fires, the fallback did not take effect."
)

for r in natives:
    assert r["balance"] > 0, f"native {r['network']} decoded to zero"

priced = [r for r in rows if r["price_usd"] is not None]
print(f"CHECK native decode ....... PASS ({len(natives)} native positions)")
print(f"CHECK price field parsed .. {'PASS' if priced else 'FAIL'} "
      f"({len(priced)}/{len(rows)} positions priced)")

if not priced:
    print("\n!! withPrices returned nothing usable. Inspect the RAW SAMPLE above")
    print("!! and extend parse_price() before continuing — every downstream")
    print("!! ratio depends on this.")
if unknowns:
    print(f"\nundecodable non-native positions: {len(unknowns)} "
          "(missing decimals in metadata — almost always spam)")

## 3. Spam filter

From the `reference.ipynb` output:

```
Visit liquid-eth.org claim rewards   115,792,089,237,316,195,423,570,985,...
```

That is `uint256` max. One row like it and every `portfolio_ratio` collapses to
approximately zero. The same output also shows a scam cluster sharing one
identical balance — `26,905,118.833873` appeared as SEGA, Nestle, Motorola,
Patek Philippe, Valve Software and Peak Games.

The spec says unknown assets must not be treated as stablecoins. They also must
not be counted in portfolio value at all.

In [ ]:
UINT256_MAX = 2 ** 256 - 1
ABSURD_BALANCE = Decimal("1e15")

SPAM_TEXT = [
    r"https?://", r"www\.", r"\.(com|eu|cc|org|io|xyz|net|ru|top|site)\b",
    r"\bclaim\b", r"\bvisit\b", r"\breward", r"\bairdrop\b",
    r"\bbot\b", r"@", r"\bt\.me\b", r"\bfree\b", r"\bgift\b",
]
SPAM_RE = re.compile("|".join(SPAM_TEXT), re.I)

# Scam contracts are frequently minted in batches with one shared supply,
# so an exact balance repeated across unrelated symbols is a strong tell.
balance_counts = Counter(
    str(r["balance"]) for r in rows if r["balance"] is not None
)


def spam_reasons(row):
    reasons = []
    text = f"{row.get('symbol') or ''} {row.get('name') or ''}"

    if SPAM_RE.search(text):
        reasons.append("url/keyword in name")
    if len(row.get("symbol") or "") > 12:
        reasons.append("symbol too long")
    if any(ord(c) > 127 for c in text):
        reasons.append("non-ascii in name")
    if row["raw"] == UINT256_MAX:
        reasons.append("uint256-max balance")
    if row["balance"] is not None and row["balance"] > ABSURD_BALANCE:
        reasons.append("absurd supply")
    if row["undecodable"]:
        reasons.append("no decimals in metadata")
    if row["price_usd"] is None:
        reasons.append("no price feed")
    if (row["balance"] is not None
            and balance_counts[str(row["balance"])] > 1):
        reasons.append("duplicate-balance cluster")
    return reasons


for r in rows:
    r["spam_reasons"] = spam_reasons(r)
    # "no price feed" alone is weak evidence; require it to be corroborated.
    hard = [x for x in r["spam_reasons"] if x != "no price feed"]
    r["is_spam"] = bool(hard)

clean = [r for r in rows if not r["is_spam"]]
spam = [r for r in rows if r["is_spam"]]

print(f"clean positions: {len(clean)}   filtered as spam: {len(spam)}")
print("\n--- FILTERED (first 15) ---")
for r in spam[:15]:
    sym = (r["symbol"] or "?")[:28]
    print(f"  {sym:<30} {', '.join(r['spam_reasons'])}")

print("\n--- KEPT ---")
for r in sorted(clean, key=lambda x: -(x["value_usd"] or 0)):
    val = f"${r['value_usd']:,.2f}" if r["value_usd"] is not None else "no price"
    print(f"  {r['network']:<16} {(r['symbol'] or '?'):<10} "
          f"{r['balance']:>20,.6f}  {val:>16}")

### CHECK — spam filter

In [ ]:
assert not any(r["raw"] == UINT256_MAX for r in clean), \
    "uint256-max position survived the filter"

assert not any(
    SPAM_RE.search(f"{r.get('symbol') or ''} {r.get('name') or ''}")
    for r in clean
), "a URL/keyword spam token survived the filter"

total_clean = sum(r["value_usd"] or 0 for r in clean)
total_all = sum(r["value_usd"] or 0 for r in rows)

print("CHECK no uint256-max in clean set .... PASS")
print("CHECK no url-spam in clean set ....... PASS")
print(f"\nportfolio value (filtered):   ${total_clean:,.2f}")
print(f"portfolio value (unfiltered): ${total_all:,.2f}")
assert total_clean > 0, "filtered portfolio has zero value — filter is too aggressive"


## 4. Classification

The spec requires three buckets — `volatile`, `stablecoin`, `unknown` — and says
classification must not rely on token names alone. So: match on **contract
address** first, then require the price to actually sit near \$1 before a token
is trusted as stable.

The registry below is checked against live Alchemy metadata in the next cell
rather than taken on faith.

In [ ]:
# Contract addresses — VERIFY against the self-check output below before
# this registry goes anywhere near production.
STABLE_REGISTRY = {
    "eth-mainnet": {
        "0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48": "USDC",
        "0xdac17f958d2ee523a2206206994597c13d831ec7": "USDT",
        "0x6b175474e89094c44da98b954eedeac495271d0f": "DAI",
    },
    "base-mainnet": {
        "0x833589fcd6edb6e08f4c7c32d4f71b54bda02913": "USDC",
    },
    "arb-mainnet": {
        "0xaf88d065e77c8cc2239327c5edb3a432268e5831": "USDC",
    },
    "polygon-mainnet": {
        "0x2791bca1f2de4661ed88a30c99a7a9449aa84174": "USDC.e",
    },
}

STABLE_SYMBOLS = {"USDC", "USDT", "DAI", "USDC.E", "FDUSD", "PYUSD", "USDE", "TUSD"}
PEG_TOLERANCE = 0.05  # a depegged stable is a risk event, not a stable


def classify(row):
    if row["is_spam"]:
        return "unknown", "filtered as spam"

    addr = (row.get("address") or "").lower()
    symbol = (row.get("symbol") or "").upper()
    price = row.get("price_usd")

    known_stable = (
        addr in STABLE_REGISTRY.get(row["network"], {})
        or symbol in STABLE_SYMBOLS
    )

    if known_stable:
        if price is None:
            return "unknown", "stable candidate with no price"
        if abs(price - 1.0) > PEG_TOLERANCE:
            return "volatile", f"stable candidate depegged at ${price:.4f}"
        return "stablecoin", "registry/symbol match, price near peg"

    if price is None:
        return "unknown", "no price feed"

    return "volatile", "priced, non-stable"


for r in rows:
    r["classification"], r["classification_reason"] = classify(r)

buckets = defaultdict(list)
for r in rows:
    buckets[r["classification"]].append(r)

for name in ("volatile", "stablecoin", "unknown"):
    group = buckets[name]
    value = sum(x["value_usd"] or 0 for x in group)
    print(f"{name:<12} {len(group):>3} positions   ${value:>16,.2f}")

### CHECK — classification & exposure

In [ ]:
assert not any(
    r["classification"] == "stablecoin" for r in rows if r["is_spam"]
), "spam token classified as a stablecoin — the exact failure the spec warns about"

stable_value = sum(r["value_usd"] or 0 for r in buckets["stablecoin"])
volatile_value = sum(r["value_usd"] or 0 for r in buckets["volatile"])
portfolio_value = stable_value + volatile_value

assert portfolio_value > 0, "portfolio has no classifiable value"

stable_ratio = stable_value / portfolio_value
volatile_ratio = volatile_value / portfolio_value

assert abs(stable_ratio + volatile_ratio - 1.0) < 1e-9, "ratios do not sum to 1"

print("CHECK no spam classified as stable .. PASS")
print("CHECK ratios sum to 1 ............... PASS")
print(f"\nportfolio value:    ${portfolio_value:,.2f}")
print(f"volatile exposure:  {volatile_ratio:6.2%}")
print(f"stable exposure:    {stable_ratio:6.2%}")

if not buckets["stablecoin"]:
    print("\n!! NO STABLECOINS IN THIS WALLET.")
    print("!! The stablecoin branch of classify() is still unproven — this was")
    print("!! true of reference.ipynb too. Re-run against a wallet holding USDC")
    print("!! before trusting the allocation engine.")

### Registry self-check

Confirms each hardcoded address really is the token we labelled it, using
Alchemy's own metadata. Run this once; treat a mismatch as a hard blocker.

In [ ]:
def verify_registry():
    checked = failed = 0

    for network, entries in STABLE_REGISTRY.items():
        if network not in NETWORKS:
            continue

        for address, expected in entries.items():
            resp = requests.post(
                f"https://{network}.g.alchemy.com/v2/{ALCHEMY_API_KEY}",
                json={
                    "id": 1, "jsonrpc": "2.0",
                    "method": "alchemy_getTokenMetadata",
                    "params": [address],
                },
                timeout=30,
            )
            if resp.status_code != 200:
                print(f"  {network:<16} {address}  HTTP {resp.status_code}")
                continue

            meta = (resp.json().get("result") or {})
            actual = (meta.get("symbol") or "").upper()
            checked += 1

            ok = actual and actual.rstrip(".E") in expected.upper().rstrip(".E")
            if not ok:
                failed += 1
            print(f"  {'OK ' if ok else 'BAD'} {network:<16} {expected:<8} "
                  f"-> reported as '{meta.get('symbol')}'")

    print(f"\nchecked {checked}, mismatched {failed}")
    assert failed == 0, "registry contains a wrong contract address — fix before use"


print("--- STABLECOIN REGISTRY SELF-CHECK ---")
verify_registry()

## 5. Polymarket

`/events` is dropped — scanning 500 events by keyword surfaced 7 markets in
`reference.ipynb`. `/public-search` returns better-targeted, higher-liquidity
results.

The per-asset `if/elif` chain is replaced by config, and `wallet_assets` is now
actually built from the wallet step instead of being referenced undefined.

In [ ]:
SEARCH_URL = "https://gamma-api.polymarket.com/public-search"
PRICE_URL = "https://clob.polymarket.com/prices-history"

ASSET_CONFIG = {
    "ETH":   {"terms": ["ethereum", "ether"], "patterns": [r"\bethereum\b", r"\bether\b"]},
    "WETH":  {"terms": ["ethereum"],          "patterns": [r"\bethereum\b"]},
    "BTC":   {"terms": ["bitcoin"],           "patterns": [r"\bbitcoin\b", r"\bbtc\b"]},
    "WBTC":  {"terms": ["bitcoin"],           "patterns": [r"\bbitcoin\b", r"\bbtc\b"]},
    "SOL":   {"terms": ["solana"],            "patterns": [r"\bsolana\b"]},
    "DOGE":  {"terms": ["dogecoin"],          "patterns": [r"\bdogecoin\b"]},
    "XRP":   {"terms": ["xrp", "ripple"],     "patterns": [r"\bxrp\b", r"\bripple\b"]},
    "ADA":   {"terms": ["cardano"],           "patterns": [r"\bcardano\b"]},
    "BNB":   {"terms": ["bnb", "binance"],    "patterns": [r"\bbnb\b"]},
    "AVAX":  {"terms": ["avalanche"],         "patterns": [r"\bavalanche\b"]},
    "LINK":  {"terms": ["chainlink"],         "patterns": [r"\bchainlink\b"]},
    "SUI":   {"terms": ["sui"],               "patterns": [r"\bsui\b"]},
    "OP":    {"terms": ["optimism"],          "patterns": [r"\boptimism\b"]},
    "POL":   {"terms": ["polygon"],           "patterns": [r"\bpolygon\b"]},
    "MATIC": {"terms": ["polygon"],           "patterns": [r"\bpolygon\b"]},
    "CAKE":  {"terms": ["pancakeswap"],       "patterns": [r"\bpancakeswap\b"]},
    "SHIB":  {"terms": ["shiba"],             "patterns": [r"\bshiba\b"]},
    "PEPE":  {"terms": ["pepe"],              "patterns": [r"\bpepe\b"]},
    "MOG":   {"terms": ["mog"],               "patterns": [r"\bmog\b"]},
}

# THE FIX for the undefined name: derive it from the wallet.
wallet_assets = sorted({
    (r["symbol"] or "").upper()
    for r in rows
    if r["classification"] == "volatile"
})

covered = [a for a in wallet_assets if a in ASSET_CONFIG]
uncovered = [a for a in wallet_assets if a not in ASSET_CONFIG]

print("volatile assets in wallet:", len(wallet_assets))
print("with Polymarket config:   ", covered)
print("without config:           ", uncovered[:20])
print(f"\ncoverage: {len(covered)}/{len(wallet_assets)} "
      f"({len(covered)/max(len(wallet_assets),1):.0%})")
print("\nAssets with no prediction market need a fallback (e.g. beta-to-ETH).")

In [ ]:
def parse_json_field(value):
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return value
    return value


def extract_yes(market):
    outcomes = parse_json_field(market.get("outcomes", []))
    prices = parse_json_field(market.get("outcomePrices", []))
    token_ids = parse_json_field(market.get("clobTokenIds", []))

    if not isinstance(outcomes, list):
        return None

    for i, outcome in enumerate(outcomes):
        if str(outcome).strip().lower() != "yes":
            continue
        if i >= len(token_ids):
            return None
        probability = None
        if i < len(prices):
            try:
                probability = float(prices[i])
            except (TypeError, ValueError):
                pass
        return {"yes_token": token_ids[i], "probability": probability}
    return None


def get_change_24h(token_id):
    """reference.ipynb used a 300s window at fidelity=1 and got +0.00% on
    nearly every market — low-liquidity books simply have no ticks that
    minute. A day at hourly fidelity actually moves."""
    now = int(time.time())
    resp = requests.get(
        PRICE_URL,
        params={"market": token_id, "startTs": now - 86400,
                "endTs": now, "fidelity": 60},
        timeout=20,
    )
    if resp.status_code != 200:
        return None
    history = resp.json().get("history", [])
    if len(history) < 2:
        return None
    try:
        first, last = float(history[0]["p"]), float(history[-1]["p"])
        return None if first == 0 else (last - first) / first
    except (TypeError, ValueError, KeyError):
        return None


def search_markets(asset, max_markets=12):
    config = ASSET_CONFIG[asset]
    compiled = [re.compile(p, re.I) for p in config["patterns"]]
    results, seen = [], set()

    for term in config["terms"]:
        resp = requests.get(
            SEARCH_URL,
            params={"q": term, "limit_per_type": 20, "events_status": "active"},
            timeout=30,
        )
        resp.raise_for_status()
        data = resp.json()

        for event in data.get("events", []):
            event_title = event.get("title", "") or ""
            event_text = f"{event_title} {event.get('description', '') or ''}"

            for market in event.get("markets", []):
                if market.get("closed"):
                    continue

                question = market.get("question", "") or ""
                text = f"{question} {event_text}"

                if not any(rx.search(text) for rx in compiled):
                    continue

                mid = market.get("id")
                if mid in seen:
                    continue
                seen.add(mid)

                yes = extract_yes(market)
                if yes is None or yes["probability"] is None:
                    continue

                def as_float(v):
                    try:
                        return float(v or 0)
                    except (TypeError, ValueError):
                        return 0.0

                results.append({
                    "question": question,
                    "event": event_title,
                    "end_date": (market.get("endDate")
                                 or event.get("endDate") or "")[:10],
                    "probability": yes["probability"],
                    "yes_token": yes["yes_token"],
                    "volume": as_float(market.get("volume")),
                    "liquidity": as_float(market.get("liquidity")),
                })

    results.sort(key=lambda x: (x["liquidity"], x["volume"]), reverse=True)
    return results[:max_markets]


polymarket_data = {}
for asset in covered:
    markets = search_markets(asset)
    polymarket_data[asset] = markets
    print(f"{asset:<6} {len(markets):>3} markets"
          f"   top liquidity ${markets[0]['liquidity']:,.0f}" if markets
          else f"{asset:<6}   0 markets")

### CHECK — 24h change

The `change_5m` column in `reference.ipynb` read `+0.00%` on nearly every
market. A 300-second window at `fidelity=1` has no ticks on a book with a few
hundred dollars of liquidity.

In [ ]:
# Validate the 24h-change fix. In reference.ipynb this column read +0.00%
# on almost every row, which made it useless as a signal.

change_samples = []

for asset, markets in polymarket_data.items():
    for market in markets[:3]:
        change = get_change_24h(market["yes_token"])
        market["change_24h"] = change
        change_samples.append(change)
        shown = f"{change:+.2%}" if change is not None else "no history"
        print(f"{asset:<6} {shown:>12}   {market['question'][:58]}")

observed = [c for c in change_samples if c is not None]
flat = [c for c in observed if abs(c) < 1e-9]
flat_ratio = len(flat) / len(observed) if observed else 1.0

print(f"\nsampled: {len(change_samples)}   with history: {len(observed)}"
      f"   flat: {len(flat)} ({flat_ratio:.0%})")

if flat_ratio > 0.8:
    print("\n!! Still mostly flat. Widen to 7d (startTs = now - 604800,")
    print("!! fidelity=180) or restrict to markets above ~$10k liquidity.")
else:
    print("\nCHECK 24h change is informative .... PASS")

## 6. Market-implied CDF

This is the part `reference.ipynb` produced without naming it.

Each `dip to $X` market is `P(price <= X)`. Each `reach $X` market is
`P(price >= X)`, so `1 - p` is also a point on the same CDF. A family of these
markets on one asset at one expiry is a **market-implied cumulative
distribution**, backed by real money.

That is not a nudge factor for a volatility parameter — it is a distribution the
Monte Carlo can be calibrated against directly.

**Expiry matters.** `Dec 31 2026` and `Sep 30 2026` are different random
variables, so points are only comparable within one expiry bucket. Mixing them
produces a curve that looks fine and means nothing.

In [ ]:
DIP_RE = re.compile(
    r"\b(?:dip|fall|drop|decline|below)\s+(?:to\s+)?\$?([\d,]+(?:\.\d+)?)\s*([kKmM]?)"
)
REACH_RE = re.compile(
    r"\b(?:reach|hit|above|exceed|surpass)\s+\$?([\d,]+(?:\.\d+)?)\s*([kKmM]?)"
)

MULT = {"k": 1_000, "m": 1_000_000, "": 1}


def parse_threshold(question):
    """-> (threshold, 'le'|'ge') or None."""
    for regex, side in ((DIP_RE, "le"), (REACH_RE, "ge")):
        match = regex.search(question)
        if match:
            amount = float(match.group(1).replace(",", ""))
            return amount * MULT[match.group(2).lower()], side
    return None


def build_cdf(markets):
    """Group by expiry, convert each market to P(X <= threshold)."""
    by_expiry = defaultdict(list)

    for market in markets:
        parsed = parse_threshold(market["question"])
        if parsed is None:
            continue
        threshold, side = parsed
        p = market["probability"]
        cumulative = p if side == "le" else 1.0 - p

        by_expiry[market["end_date"]].append({
            "threshold": threshold,
            "p_le": cumulative,
            "side": side,
            "question": market["question"],
            "liquidity": market["liquidity"],
        })

    for expiry in by_expiry:
        by_expiry[expiry].sort(key=lambda x: x["threshold"])
    return dict(by_expiry)


def check_monotonic(points):
    """A CDF must be non-decreasing. Violations = stale quotes or a bad parse."""
    violations = []
    for prev, curr in zip(points, points[1:]):
        if curr["p_le"] < prev["p_le"] - 1e-9:
            violations.append((prev, curr))
    return violations


cdfs = {}
for asset, markets in polymarket_data.items():
    cdf = build_cdf(markets)
    if not cdf:
        continue
    cdfs[asset] = cdf

    print(f"\n{'=' * 68}\n{asset}\n{'=' * 68}")
    for expiry, points in sorted(cdf.items()):
        if len(points) < 2:
            continue
        print(f"\n  expiry {expiry}   ({len(points)} points)")
        print(f"  {'threshold':>14}  {'P(X<=t)':>9}  {'from':>4}  liquidity")
        for pt in points:
            print(f"  ${pt['threshold']:>13,.0f}  {pt['p_le']:>8.2%}  "
                  f"{pt['side']:>4}  ${pt['liquidity']:>12,.0f}")

        bad = check_monotonic(points)
        if bad:
            print(f"  !! {len(bad)} monotonicity violation(s):")
            for prev, curr in bad:
                print(f"     ${prev['threshold']:,.0f} @ {prev['p_le']:.2%}"
                      f"  ->  ${curr['threshold']:,.0f} @ {curr['p_le']:.2%}")
        else:
            print("  monotonic: OK")

### CHECK — CDF quality

A usable calibration target needs at least three points at one expiry and a
roughly monotonic curve.

In [ ]:
usable = {}

for asset, cdf in cdfs.items():
    for expiry, points in cdf.items():
        if len(points) < 3:
            continue
        violations = check_monotonic(points)
        severity = len(violations) / max(len(points) - 1, 1)
        if severity <= 0.25:
            usable.setdefault(asset, {})[expiry] = points

print("assets with a usable calibration target:", list(usable) or "NONE")

for asset, expiries in usable.items():
    for expiry, points in expiries.items():
        lo, hi = points[0], points[-1]
        print(f"  {asset} @ {expiry}: {len(points)} points, "
              f"${lo['threshold']:,.0f} ({lo['p_le']:.1%}) -> "
              f"${hi['threshold']:,.0f} ({hi['p_le']:.1%})")

assert usable, (
    "No asset produced a usable CDF. Either the wallet holds nothing with a "
    "prediction market, or parse_threshold() needs more question shapes."
)
print("\nCHECK usable CDF exists .... PASS")

## 7. Validated payload

One structure, every field checked above. This is what the quant engine should
consume — not raw API responses.

Note what is deliberately absent: no allocation percentage. Per the spec, the
quantitative engine produces the number and the LLM only explains it.

In [ ]:
def to_jsonable(value):
    return float(value) if isinstance(value, Decimal) else value


payload = {
    "wallet_address": WALLET_ADDRESS,
    "generated_at": int(time.time()),
    "portfolio": {
        "total_value_usd": round(portfolio_value, 2),
        "volatile_ratio": round(volatile_ratio, 6),
        "stable_ratio": round(stable_ratio, 6),
        "positions": [
            {
                "symbol": r["symbol"],
                "network": r["network"],
                "address": r["address"],
                "quantity": to_jsonable(r["balance"]),
                "price_usd": r["price_usd"],
                "value_usd": round(r["value_usd"], 2) if r["value_usd"] else None,
                "portfolio_ratio": round((r["value_usd"] or 0) / portfolio_value, 6),
                "classification": r["classification"],
            }
            for r in sorted(
                buckets["volatile"] + buckets["stablecoin"],
                key=lambda x: -(x["value_usd"] or 0),
            )
        ],
        "excluded_positions": len(spam),
    },
    "market_signals": {
        asset: {
            "markets": [
                {k: m[k] for k in
                 ("question", "end_date", "probability", "liquidity", "volume")}
                for m in markets
            ],
            "implied_cdf": {
                expiry: [
                    {"threshold": p["threshold"], "p_le": round(p["p_le"], 6)}
                    for p in points
                ]
                for expiry, points in usable.get(asset, {}).items()
            },
        }
        for asset, markets in polymarket_data.items() if markets
    },
    "coverage": {
        "volatile_assets": len(wallet_assets),
        "with_prediction_markets": len(covered),
        "calibratable": len(usable),
    },
}

serialized = json.dumps(payload, indent=2, default=str)

# ---- final assertions ----
ratios = [p["portfolio_ratio"] for p in payload["portfolio"]["positions"]]
assert abs(sum(ratios) - 1.0) < 1e-4, f"position ratios sum to {sum(ratios)}"
assert all(0 <= r <= 1 for r in ratios), "a portfolio ratio is out of bounds"
assert payload["portfolio"]["total_value_usd"] > 0

for asset, block in payload["market_signals"].items():
    for market in block["markets"]:
        assert 0 <= market["probability"] <= 1, \
            f"{asset}: probability out of bounds"

print("CHECK ratios sum to 1 ......... PASS")
print("CHECK ratios in [0,1] ......... PASS")
print("CHECK probabilities in [0,1] .. PASS")
print(f"\npayload: {len(serialized):,} bytes")
print(serialized[:1500])

with open("validated_payload.json", "w") as fh:
    fh.write(serialized)
print("\nwrote validated_payload.json")

## 8. Summary

In [ ]:
print("=" * 62)
print("WALLERINA API VALIDATION")
print("=" * 62)

lines = [
    ("Alchemy reachable",            True),
    ("Native balances decoded",      bool(natives)),
    ("USD prices parsed",            bool(priced)),
    ("Spam filtered",                bool(spam)),
    ("Volatile assets classified",   bool(buckets["volatile"])),
    ("Stablecoins present & classified", bool(buckets["stablecoin"])),
    ("Polymarket search working",    any(polymarket_data.values())),
    ("24h change is informative",    (flat_ratio < 0.8) if observed else None),
    ("Usable implied CDF",           bool(usable)),
]

for label, ok in lines:
    mark = "PASS" if ok else ("WARN" if ok is None else "FAIL")
    print(f"  {label:<38} {mark}")

print("=" * 62)

if not buckets["stablecoin"]:
    print("\nBLOCKER: no stablecoin in the test wallet — the stable branch of")
    print("the classifier is still unproven. Re-run against a USDC-holding")
    print("wallet before building the allocation engine on top of this.")

print(f"\nCalibration targets available for: {list(usable) or 'none'}")
print("Next: fit the Monte Carlo so simulated paths reproduce these")
print("tail probabilities, instead of running GBM on historical vol alone.")